# Draw Steel Noticeboard: Technical Roadmap & Schema

**Project:** Unified Production Roadmap for Multi-Instance Noticeboard
**Stack:** Supabase (DB), FastAPI (Backend), Discord Bot (Controller), React (Frontend)

---

## 1. System Architecture Overview

The system links a Discord Bot and a React Frontend to a shared Supabase database via a FastAPI backend[cite: 1].

### Core Components
* **Supabase:** Handles PostgreSQL database, Auth (Discord/Google), and Storage[cite: 1].
* **FastAPI:** Serves as the logic layer to validate complex character JSONs and manage noticeboard instances[cite: 1, 234].
* **Discord Bot:** Acts as the Admin interface for creating instances and the User interface for posting notices[cite: 234, 236].
* **React Frontend:** Provides a dashboard for OAuth login and character sheet imports[cite: 235].

## 2. Supabase Data Model (SQL)

Execute the following SQL in your Supabase SQL Editor. This schema supports multi-tenancy (instances) and unstructured character data (JSONB)[cite: 1, 236].

### Key Features
* **`instances`**: Separation of concerns for different Discord guilds[cite: 236].
* **`characters`**: Uses `JSONB` to store the deep nesting of Draw Steel export files (like the provided Boris character)[cite: 237, 1].

In [ ]:
-- Enable UUIDs
create extension if not exists "uuid-ossp";

-- 1. INSTANCES (Admin-created Boards)
create table public.instances (
  id uuid default uuid_generate_v4() primary key,
  name text not null,
  created_by uuid references auth.users(id),
  created_at timestamp with time zone default timezone('utc'::text, now()) not null
);

-- 2. DISCORD MAPPING (Bot Integration)
create table public.discord_instances (
  id uuid default uuid_generate_v4() primary key,
  instance_id uuid references public.instances(id) on delete cascade,
  guild_id bigint not null unique -- The Discord Server ID
);

-- 3. CHARACTERS (JSONB Storage for .ds-hero files)
create table public.characters (
  id uuid default uuid_generate_v4() primary key,
  user_id uuid references auth.users(id) on delete cascade,
  name text not null, 
  data jsonb not null, -- Stores the full JSON payload (e.g., Boris character)
  created_at timestamp with time zone default timezone('utc'::text, now()) not null
);

-- Indexing for efficient querying of Character Data
create index idx_characters_data on public.characters using gin (data);

-- 4. NOTICES (The actual posts)
create table public.notices (
  id uuid default uuid_generate_v4() primary key,
  instance_id uuid references public.instances(id) on delete cascade,
  author_id uuid references public.characters(id), -- Posting as a Character
  content text not null,
  timestamp timestamp with time zone default timezone('utc'::text, now()) not null
);

## 3. Backend Implementation (FastAPI)

### Endpoints Required
1.  **`POST /instances`**: Admin-only endpoint. Triggers creation of a row in the `instances` table[cite: 1, 236].
2.  **`POST /characters`**: 
    * Accepts `.ds-hero` JSON upload.
    * **Validation Logic**: Must parse `ancestry`, `class`, and `name` from the JSON structure (e.g., validating "Revenant" or "Null" from the Boris file)[cite: 1, 237].
3.  **`GET /notices/{instance_id}`**: Retrieves posts for the specific board/Discord server.

## 4. Discord Bot Logic

### Admin Commands
* `/register_instance`: Checks if the user is an Admin, then creates a new Instance in Supabase and maps the current Guild ID to it[cite: 236].

### User Workflow
* **Posting**: Users select a character (stored in `characters`) to post a notice.
* **Lookup**: The bot queries Supabase for characters linked to the user's Discord ID (via OAuth link)[cite: 1].

## 5. React Frontend & OAuth

* **Auth**: Implement Supabase Auth for **Discord** and **Google** (Gmail)[cite: 235].
* **Character Sheet Import**: 
    * Drag-and-drop interface for `.ds-hero` files.
    * Parses the complex nested JSON (e.g., `ancestry.features`) before sending to the Backend[cite: 1, 237].